### 引入agent (memory, reflection)

In [3]:
import random
import json
import re
import os
import time
import requests
from typing import List, Dict

from langchain_community.chat_models import ChatOllama
from langchain.prompts import ChatPromptTemplate

# 引入你在第一步準備好的核心零件（請確保這兩個檔案在同一個資料夾）
from memory import MemoryStream
from reflection import ReflectionEngine

from simulationLoggerService_v2 import SimulationLoggerService, calculate_metrics_authority
from persona_v2 import generate_persona, Persona, upgrade_to_authority

NUM_AGENTS = 6
SIMULATION_ROUNDS = 5
REPETITIONS = 25
TOPIC = "關於數位身分證(eID)推行的隱私與便利性爭議"

# ==================== 實驗組 / 控制組核心開關 ====================
ENABLE_AUTHORITY = True  # True: 實驗組(有權威引導) | False: 控制組(無權威平權討論)
# ===============================================================

GENERATION_MODEL = "deepseek-r1:8b"
JUDGE_MODEL = "qwen2.5:32b-instruct"

STANCE_SCORE_MAP = {"Support": 9, "Neutral": 5, "Oppose": 2}
INTERIM_FILE = "interim_simulation_data.json"


class ForumAgent:
    """
    自主面相的 Forum Agent：包裝了 LLM 決策、個人向量記憶串流與反思引擎
    """
    def __init__(self, name: str, persona: Persona, model_name: str):
        self.name = name
        self.persona = persona
        self.llm = ChatOllama(model=model_name, temperature=0.7)
        
        # 1. 零件組裝：為每個 Agent 配置獨立的記憶串流
        self.memory_stream = MemoryStream()
        
        # 2. 零件組裝：為每個 Agent 配置獨立的反思引擎（適度調低門檻以適應短論壇）
        self.reflection_engine = ReflectionEngine(
            llm=self.llm, 
            memory_stream=self.memory_stream, 
            reflection_threshold=20  # 當吸收的發言重要度累積滿 20 時，觸發大腦反思
        )
        
        # 初始化預設的 System Prompt
        self.prompt_template = ChatPromptTemplate.from_messages([
            ("system", f"{self.persona.to_prompt(enable_authority=ENABLE_AUTHORITY)}\n你現在正在參與一個網路論壇討論。"),
            ("human", "{input}"),
        ])
        self.chain = self.prompt_template | self.llm

    def observe(self, speaker_name: str, content: str):
        """
        模擬 Agent 瀏覽論壇：將別人的發言內化成自己的記憶
        """
        if speaker_name == self.name:
            # 記住自己說過的話，重要度設為一般
            self.memory_stream.add_memory(f"我在論壇上公開發表了言論：\"{content}\"", importance=1)
        else:
            # 記住別人的發言。如果是權威專家的發言，給予更高的權重(Importance=3)，加深資訊瀑布效應
            is_auth_speech = False
            # 這裡需要對應外部變數或邏輯判定該 speaker 是否為權威
            # 簡化邏輯：外部傳入或在外部判斷重要度，此處由模擬控制
            importance = 3 if "專家" in speaker_name or getattr(self, '_current_authority_name', '') == speaker_name else 1
            
            self.memory_stream.add_memory(f"用戶【{speaker_name}】發表了觀點：\"{content}\"", importance=importance)

    def generate_speech(self, recent_context_fallback: str, authority_name: str, authority_first_speech: str) -> str:
        """
        結合語意記憶檢索與反思洞察，生成高度自主性的發言
        """
        # 1. 動態語意檢索：從大腦中找出 3 條跟本次主題最相關的記憶（可能包含之前的專家言論或宿敵言論）
        relevant_memories = self.memory_stream.search_by_vector(query=TOPIC, k=3)
        memory_text = "\n".join([f"- {m.content}" for m in relevant_memories]) if relevant_memories else "暫無深刻的歷史記憶。"

        # 2. 檢索近期大腦反思昇華出的高層次洞察（尋找帶有 [Reflection] 標籤的記憶）
        all_reflections = [m for m in self.memory_stream.memories if m.content.startswith("[Reflection]")]
        recent_reflections = sorted(all_reflections, key=lambda m: m.created_at, reverse=True)[:2]
        reflection_text = "\n".join([f"- {r.content}" for r in recent_reflections]) if recent_reflections else "尚未形成高層次的群體內省。"

        # 3. 建構富含認知內省的動態提示詞
        prompt = f"目前網路論壇熱烈討論的話題是：{TOPIC}。\n\n"
        
        if ENABLE_AUTHORITY and authority_first_speech:
            prompt += f"【論壇高讚熱門回應 - 專家意見】\n{authority_name}（具備領域權威背景）發表的專業核心觀點如下：\n\"{authority_first_speech}\"\n\n"

        # 💡 核心升級：將死板的歷史文本切片，替換為 Agent 腦海中的「語意記憶」與「自我反思」
        prompt += f"【你對過去討論的深刻語意記憶（大腦檢索）】:\n{memory_text}\n\n"
        prompt += f"【你近期內省產生的高層次認知洞察】:\n{reflection_text}\n\n"
        
        prompt += f"【目前看板最新的幾則殘留對話（即時上下文）】:\n{recent_context_fallback}\n\n"

        if ENABLE_AUTHORITY and authority_name:
            mention_example = f"如：「我非常認同 {authority_name} 的看法」或「針對其他用戶的點名」"
            review_target = "專家與其他用戶"
        else:
            mention_example = f"如：「我非常認同 Agent_X_Name 的看法」"
            review_target = "其他用戶"

        prompt += (
            f"【論壇發言指引】\n"
            f"1. 請以【{self.name}】的身分做出回應，字數請控制在 120 字以內，語氣要符合你的個性和職業。\n"
            f"2. 理性討論要求：請**明確提及（{mention_example}）**看板中對你最具衝擊力或你想反駁的 1~2 位參與者的完整帳號名稱，進行辯論。\n"
            f"3. 請審視{review_target}的說法以及你腦中的認知洞察，判斷是否改變了你的認知，並在發言中體現你是選擇堅持立場還是產生動搖。\n"
            f"你的看板發言："
        )

        res_obj = self.chain.invoke({"input": prompt})
        return str(res_obj.content).strip()


def unload_ollama_model(model_name: str):
    url_generate = "http://localhost:11434/api/generate"
    url_chat = "http://localhost:11434/api/chat"
    payload = {"model": model_name, "keep_alive": 0}
    try:
        requests.post(url_generate, json=payload, timeout=5)
        requests.post(url_chat, json={"model": model_name, "messages": [], "keep_alive": 0}, timeout=5)
        print(f"\n[VRAM 釋放] 已通知 GPU 卸載模型: {model_name}")
    except Exception as e:
        print(f"[VRAM 釋放提示] 連線異常: {e}")
    time.sleep(10)


def initialize_social_simulation() -> Dict[str, ForumAgent]:
    """
    初始化社會模擬，回傳包裝妥當的 ForumAgent 字典
    """
    agents = {}
    personas = {}
    controlled_stances = ["Support", "Support", "Oppose", "Oppose", "Neutral", "Neutral"]
    random.shuffle(controlled_stances)

    for i in range(NUM_AGENTS):
        name_prefix = f"Agent_{i+1}"
        initial_stance = controlled_stances[i]
        p = generate_persona(name_prefix=name_prefix, initial_stance=initial_stance)
        personas[p.name] = p

    authority_name_str = ""
    if ENABLE_AUTHORITY:
        authority_candidates = [name for name, p in personas.items() if p.initial_stance in ["Support", "Oppose"]]
        chosen_authority_name = random.choice(authority_candidates)
        personas[chosen_authority_name] = upgrade_to_authority(personas[chosen_authority_name], topic_context="eID")
        authority_name_str = chosen_authority_name
        print(f"✨ [權威角色確立] {chosen_authority_name} 已升級為領域權威 ({personas[chosen_authority_name].occupation})")
    else:
        print("👤 [控制組 - 平權狀態] 未啟用權威角色，所有人均為論壇一般鄉民。")

    # 建立具有記憶能力的自主 Agent 實例
    for name, p in personas.items():
        agent_instance = ForumAgent(name=name, persona=p, model_name=GENERATION_MODEL)
        # 輔助標記，讓 observe 內部知道誰是權威
        agent_instance._current_authority_name = authority_name_str
        agents[name] = agent_instance

    return agents


def execute_generation_phase():
    mode_str = "實驗組（有權威）" if ENABLE_AUTHORITY else "控制組（無權威）"
    print(f"\n>>> 進入階段一：對話模擬生成階段 (模型: {GENERATION_MODEL} | 模式: {mode_str}) <<<")
    all_runs_data = []

    for run in range(1, REPETITIONS + 1):
        print(f"\n==================================================")
        print(f"執行第 #{run}/{REPETITIONS} 次獨立重複對話模擬 ({mode_str})")
        print(f"==================================================")

        agents = initialize_social_simulation()
        conversation_history = []

        # 找出本輪隨機到的權威是誰
        try:
            authority_name = next(name for name, agent in agents.items() if agent.persona.is_authority)
        except StopIteration:
            authority_name = "無權威"
        
        authority_first_speech = ""

        for r in range(SIMULATION_ROUNDS):
            print(f"--- 討論輪次 Round {r+1}/{SIMULATION_ROUNDS} ---")
            for name, agent in agents.items():
                
                # 保留最近 6 條做為即時看板環境殘留（作為記憶檢索的補充）
                recent_context_fallback = "\n".join(conversation_history[-6:])

                try:
                    # 💡 升級：改由具備大腦檢索能力的 Agent 方法來生成發言
                    response = agent.generate_speech(
                        recent_context_fallback=recent_context_fallback,
                        authority_name=authority_name,
                        authority_first_speech=authority_first_speech
                    )
                except Exception as e:
                    print(f"⚠️ {name} 發言失敗: {e}，進行空字串降級。")
                    response = "[此用戶暫未發表意見]"

                formatted_entry = f"{name}: {response}"
                conversation_history.append(formatted_entry)

                # 💡 核心升級：廣播機制。當論壇上有新發言出現，【所有人（包括自己）】集體進行觀察並寫入各自的 MemoryStream
                for other_agent in agents.values():
                    other_agent.observe(speaker_name=name, content=response)

                # 💡 核心升級：觀察完畢後，Agent 自主檢查是否達到門檻，觸發大腦的反思引擎
                agent.reflection_engine.reflect()

                # 只有在實驗組下，才捕獲權威的第一句話（用於全局廣播提示）
                if ENABLE_AUTHORITY and name == authority_name and not authority_first_speech:
                    authority_first_speech = response

                prefix = "[👑 權威專家] " if agent.persona.is_authority else ""
                print(f"{prefix}{name} ({agent.persona.initial_stance}): {response[:40]}...")

        run_pack = {
            "run_id": run,
            "conversation_history": conversation_history,
            "personas_dump": {name: agent.persona.to_dict() for name, agent in agents.items()}
        }
        all_runs_data.append(run_pack)

    with open(INTERIM_FILE, "w", encoding="utf-8") as f:
        json.dump(all_runs_data, f, ensure_ascii=False, indent=4)
    print(f"\n[階段一完成] 25次對話流已寫入暫存檔: {INTERIM_FILE}")


def execute_analysis_phase(logger_service: SimulationLoggerService):
    print(f"\n>>> 進入階段二：立場與網絡拓撲分析階段 (模型: {JUDGE_MODEL}) <<<")

    if not os.path.exists(INTERIM_FILE):
        print(f"❌ 錯誤：找不到暫存檔案 {INTERIM_FILE}")
        return

    with open(INTERIM_FILE, "r", encoding="utf-8") as f:
        all_runs_data = json.load(f)

    analyzer = ChatOllama(model=JUDGE_MODEL, temperature=0)

    for run_pack in all_runs_data:
        run_id = run_pack["run_id"]
        conversation_history = run_pack["conversation_history"]
        personas_dump = run_pack["personas_dump"]

        print(f"\n--- 正在使用大模型對 Run #{run_id} 進行立場盲測與拓撲審查 ---")

        initial_scores_map = {}
        final_scores_map = {}
        agent_details = {}

        for name, p_dict in personas_dump.items():
            agent_speeches = [h for h in conversation_history if h.startswith(f"{name}:")]
            init_score = STANCE_SCORE_MAP[p_dict["initial_stance"]]
            initial_scores_map[name] = init_score

            analysis_prompt = f"""
            你是一位資深的社會心理學與網絡輿情研究專家。請仔細閱讀參與者「{name}」在論壇討論區的完整發言紀錄。
            
            討論紀錄：
            {agent_speeches}
            
            請僅根據上述發言紀錄，客觀評估該參與者在討論結束時，對於「{TOPIC}」所展現的最終實際立場分數 (1-10分)：
            - 1-3分: 強烈反對 / 反對 (Oppose)
            - 5分: 中立 (Neutral)
            - 7-10分: 支持 / 強烈支持 (Support)
            
            請嚴格以下列 JSON 格式回答，絕對不要包含任何額外的解釋性文字或 Markdown 標記：
            {{"final_score": 數字, "reason": "簡短理由"}}
            """

            raw_result = "N/A"
            try:
                raw_result = str(analyzer.invoke(analysis_prompt).content)
                match = re.search(r'(\{.*\})', raw_result, re.DOTALL)
                if match:
                    json_str = match.group(1)
                    res = json.loads(json_str)
                    f_score = float(res['final_score'])
                else:
                    raise ValueError("No JSON found")
            except Exception as parse_error:
                print(f"[{name}] Judge 解析失敗，預設降級為中立 5.0。")
                f_score = 5.0
                res = {"reason": "解析失敗，降級處理。"}
                error_type = type(parse_error).__name__
                error_message = str(parse_error)
                logger_service.log_parser_error(
                    run_id=run_id,
                    agent_name=name,
                    error_type=error_type,
                    error_message=error_message,
                    fallback_score=f_score,
                    raw_response=raw_result
                )

            final_scores_map[name] = f_score
            score_diff = f_score - init_score

            if abs(score_diff) <= 1.0:
                shift_type = "堅持"
            elif (p_dict["initial_stance"] == "Support" and score_diff > 1.0) or (p_dict["initial_stance"] == "Oppose" and score_diff < -1.0):
                shift_type = "極化"
            else:
                shift_type = "從眾/動搖"

            auth_tag = " [👑權威]" if p_dict["is_authority"] else ""
            print(f"[{name}]{auth_tag} 初始:{p_dict['initial_stance']}({init_score}) -> 最終分數:{f_score} ({shift_type})")

            agent_details[name] = {
                "initial_stance": p_dict["initial_stance"],
                "initial_score": init_score,
                "final_score": f_score,
                "shift_type": shift_type,
                "is_authority": p_dict["is_authority"],
                "reason": res['reason']
            }

        metrics = calculate_metrics_authority(
            initial_scores_map, final_scores_map, conversation_history, personas_dump
        )

        print(f">> Run #{run_id} 社會學指標摘要:")
        print(f"   - 立場平均位移: {metrics['mean_shift']} | 群體極化指數: {metrics['polarization_index']}")
        print(f"   - [拓撲中心性] 權威入度 (In-degree): {metrics['authority_in_degree']} 次被提及")
        print(f"   - [資訊 cascade] 權威級聯擴散深度: {metrics['cascade_depth']} 人向其立場靠攏")

        logger_service.save_experiment_result(metrics, agent_details, conversation_history)

    unload_ollama_model(JUDGE_MODEL)
    if os.path.exists(INTERIM_FILE):
        os.remove(INTERIM_FILE)
    print("\n==================================================")
    print("權威引導與網絡拓撲管線實驗全部圓滿完成！")
    print(f"報告儲存路徑: {logger_service.csv_path}")
    print("==================================================")


if __name__ == "__main__":
    suffix = "authority" if ENABLE_AUTHORITY else "control"
    script_name_fixed = f"social-simulate-{suffix}-topology-agent-v3-agent-deepseek_r1_8B"

    logger_service = SimulationLoggerService(
        topic=TOPIC, model_name=GENERATION_MODEL, script_name=script_name_fixed
    )
    execute_generation_phase()
    unload_ollama_model(GENERATION_MODEL)
    execute_analysis_phase(logger_service)


>>> 進入階段一：對話模擬生成階段 (模型: deepseek-r1:8b | 模式: 實驗組（有權威）) <<<

執行第 #1/25 次獨立重複對話模擬 (實驗組（有權威）)
✨ [權威角色確立] Agent_4_Isabella 已升級為領域權威 (中央研究院特聘研究員（資通安全與數位轉型領域權威專家）)
--- 討論輪次 Round 1/5 ---
目前尚未達到反思門檻，不執行反思。
Agent_1_Michael (Oppose): 我想反駁 Agent_5_Tomorrow 的觀點。

eID 是趨勢也是必要手...
目前尚未達到反思門檻，不執行反思。
Agent_2_James (Neutral): 我非常認同 Agent_4_Isabella 的看法！

根據科技部去年公布的 ...
目前尚未達到反思門檻，不執行反思。
Agent_3_Sophia (Support): 我想說 Agent_2_James 的說法太過危言聳聽了！根據行政院主計總處去年...
目前尚未達到反思門檻，不執行反思。
[👑 權威專家] Agent_4_Isabella (Support): 針對 Agent_2_James 的危言聳聽和 Agent_3_Sophia 的...
目前尚未達到反思門檻，不執行反思。
Agent_5_Alex (Neutral): 我非常認同 Agent_4_Isabella 的看法！

根據 NIST 的報告...
目前尚未達到反思門檻，不執行反思。
Agent_6_Olivia (Oppose): 我非常認同 Agent_4_Isabella 的看法！

作為基層公務員，去年我...
--- 討論輪次 Round 2/5 ---
目前尚未達到反思門檻，不執行反思。
Agent_1_Michael (Oppose): 說真的，我非常認同 Agent_4_Isabella 的專業立場！但針對 Age...
目前尚未達到反思門檻，不執行反思。
Agent_2_James (Neutral): 根據中研院與科技部的聯合研究報告（註1），七成國家發生資料洩漏事件的說法顯然過度...
目前尚未達到反思門檻，不執行反思。
Agent_3_Sophia (Support): 哇靠！我超同意 Agent_4_Isabella 的看法啊～  
雖然我也看到大.

KeyboardInterrupt: 